In [11]:
import pandas as pd

excel_file = "FSI-2023-DOWNLOAD.xlsx"

# Read all sheets
sheets = pd.read_excel(excel_file, sheet_name=None)

for sheet_name, df in sheets.items():
    print(sheet_name)
    print(df.head())

Sheet1
                     Country  Year Rank  Total  S1: Demographic Pressures  \
0                    Somalia  2023  1st  111.9                       10.0   
1                      Yemen  2023  2nd  108.9                        9.6   
2                South Sudan  2023  3rd  108.5                        9.7   
3  Congo Democratic Republic  2023  4th  107.2                        9.7   
4                      Syria  2023  5th  107.1                        7.4   

   S2: Refugees and IDPs  C3: Group Grievance  \
0                    9.0                  8.7   
1                    9.6                  8.8   
2                   10.0                  8.6   
3                    9.8                  9.4   
4                    9.1                  9.1   

   E3: Human Flight and Brain Drain  E2: Economic Inequality  E1: Economy  \
0                               8.6                      9.1          9.5   
1                               6.4                      7.9          9.9   
2   

In [12]:
documents = []

for sheet_name, df in sheets.items():

    df = df.fillna("")
    print("df",df)

    for idx, row in df.iterrows():

        row_text = ", ".join(
            [f"{col}: {row[col]}" for col in df.columns]
        )
        print("row_text",row_text)

        documents.append({
            "text": row_text,
            "metadata": {
                "sheet": sheet_name,
                "row": idx
            }
        })

df                        Country  Year   Rank  Total  S1: Demographic Pressures  \
0                      Somalia  2023    1st  111.9                       10.0   
1                        Yemen  2023    2nd  108.9                        9.6   
2                  South Sudan  2023    3rd  108.5                        9.7   
3    Congo Democratic Republic  2023    4th  107.2                        9.7   
4                        Syria  2023    5th  107.1                        7.4   
..                         ...   ...    ...    ...                        ...   
174                Switzerland  2023  175th   17.8                        2.4   
175                New Zealand  2023  176th   16.7                        1.1   
176                    Finland  2023  177th   16.0                        1.7   
177                    Iceland  2023  178th   15.7                        1.5   
178                     Norway  2023  179th   14.5                        1.4   

     S2: Refugees and ID

In [13]:

from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunked_documents = []

for doc in documents:

    chunks = text_splitter.split_text(doc["text"])

    for chunk_id, chunk in enumerate(chunks):

        chunked_documents.append({
            "text": chunk,
            "metadata": {
                **doc["metadata"],
                "chunk_id": chunk_id
            }

        })
        

print(f"Total chunks created: {len(chunked_documents)}")
print("Sample chunk:", chunked_documents)

/home/sourav/miniconda3/envs/excel/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total chunks created: 358
Sample chunk: [{'text': 'Country: Somalia, Year: 2023, Rank: 1st, Total: 111.89999999999999, S1: Demographic Pressures: 10.0, S2: Refugees and IDPs: 9.0, C3: Group Grievance: 8.7, E3: Human Flight and Brain Drain: 8.6, E2: Economic Inequality: 9.1, E1: Economy: 9.5, P1: State Legitimacy: 9.6, P2: Public Services: 9.8, P3:', 'metadata': {'sheet': 'Sheet1', 'row': 0, 'chunk_id': 0}}, {'text': 'Legitimacy: 9.6, P2: Public Services: 9.8, P3: Human Rights: 9.0, C1: Security Apparatus: 9.5, C2: Factionalized Elites: 10.0, X1: External Intervention: 9.1', 'metadata': {'sheet': 'Sheet1', 'row': 0, 'chunk_id': 1}}, {'text': 'Country: Yemen, Year: 2023, Rank: 2nd, Total: 108.89999999999999, S1: Demographic Pressures: 9.6, S2: Refugees and IDPs: 9.6, C3: Group Grievance: 8.8, E3: Human Flight and Brain Drain: 6.4, E2: Economic Inequality: 7.9, E1: Economy: 9.9, P1: State Legitimacy: 9.8, P2: Public Services: 9.6, P3:', 'metadata': {'sheet': 'Sheet1', 'row': 1, 'chunk_id'

In [4]:
!pip install langchain-text-splitters

In [6]:
!pip install langchain

In [ ]:
!python -m pip install numpy==1.26.4

In [14]:
# =========================================================
# 4. CREATE EMBEDDINGS
# =========================================================
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer(
    "BAAI/bge-small-en"
)

texts = [
    chunk["text"]
    for chunk in chunked_documents
]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True
)


/home/sourav/miniconda3/envs/excel/lib/python3.10/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12060). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5358.95it/s]


In [15]:
embeddings

array([[-0.02959093,  0.02241935, -0.00324014, ..., -0.05601116,
         0.01193031,  0.04692105],
       [-0.0255571 , -0.02253826,  0.01428468, ...,  0.02207085,
         0.03311256,  0.02242373],
       [-0.02006294, -0.00548973, -0.00808507, ..., -0.01054847,
         0.02149979,  0.06183026],
       ...,
       [-0.01929298, -0.01505974,  0.00714869, ...,  0.02293175,
         0.02626725,  0.01748393],
       [ 0.00307629,  0.01805805, -0.00677139, ..., -0.0315618 ,
         0.01187169,  0.05215962],
       [-0.02299008, -0.0274962 ,  0.00864632, ...,  0.00899522,
         0.0215491 ,  0.03439165]], dtype=float32)

In [ ]:
!python -m pip install numpy==1.26.4

In [ ]:
!python -c "import numpy; print(numpy.__version__)"

1.26.4


In [ ]:


!pip install sentence_transformers

  Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached markupsafe-3.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.m

In [ ]:
!rm -rf ~/.local/lib/python3.10/site-packages/soxr*

In [ ]:
!ls ~/.local/lib/python3.10/site-packages/ | grep soxr

In [ ]:
!python -m pip install soxr

In [17]:
import faiss
import numpy as np

In [11]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 1.4 MB/s  0:00:17a 0:00:01m eta 0:00:010m


In [18]:
# =========================================================
# 5. STORE IN FAISS VECTOR DB
# =========================================================

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
print("FAISS index initialized successfully",index)

index.add(
    embeddings.astype("float32")
)

print("FAISS index created successfully")


FAISS index initialized successfully <faiss.swigfaiss_avx512.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x7bca55c42910> >
FAISS index created successfully


In [19]:
# =========================================================
# 6. RETRIEVAL FUNCTION
# =========================================================

def retrieve(query, top_k=5):

    # Create query embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    # Search FAISS
    distances, indices = index.search(
        query_embedding.astype("float32"),
        top_k
    )

    results = []

    for idx in indices[0]:

        results.append(
            chunked_documents[idx]
        )

    return results


In [25]:
query = "what is the rank of mali?"

results = retrieve(query)

print("\nRETRIEVED RESULTS\n")

for result in results:

    print("=" * 50)

    print("Answer:", result["text"])

    print("\nMetadata:")
    print(result["metadata"])


RETRIEVED RESULTS

Answer: Country: Mali, Year: 2023, Rank: 13th, Total: 99.5, S1: Demographic Pressures: 8.8, S2: Refugees and IDPs: 8.5, C3: Group Grievance: 8.5, E3: Human Flight and Brain Drain: 7.7, E2: Economic Inequality: 7.2, E1: Economy: 7.5, P1: State Legitimacy: 8.6, P2: Public Services: 8.9, P3: Human Rights: 7.5,

Metadata:
{'sheet': 'Sheet1', 'row': 12, 'chunk_id': 0}
Answer: Country: Sudan, Year: 2023, Rank: 7th, Total: 106.19999999999999, S1: Demographic Pressures: 8.8, S2: Refugees and IDPs: 9.6, C3: Group Grievance: 9.3, E3: Human Flight and Brain Drain: 7.5, E2: Economic Inequality: 8.5, E1: Economy: 9.3, P1: State Legitimacy: 9.4, P2: Public Services: 8.6, P3:

Metadata:
{'sheet': 'Sheet1', 'row': 6, 'chunk_id': 0}
Answer: Country: Liberia, Year: 2023, Rank: 33rd, Total: 88.89999999999999, S1: Demographic Pressures: 8.4, S2: Refugees and IDPs: 7.4, C3: Group Grievance: 4.6, E3: Human Flight and Brain Drain: 6.7, E2: Economic Inequality: 7.5, E1: Economy: 8.3, P1: S

In [26]:
results = retrieve(query)

for result in results:
    print(result["text"])

Country: Mali, Year: 2023, Rank: 13th, Total: 99.5, S1: Demographic Pressures: 8.8, S2: Refugees and IDPs: 8.5, C3: Group Grievance: 8.5, E3: Human Flight and Brain Drain: 7.7, E2: Economic Inequality: 7.2, E1: Economy: 7.5, P1: State Legitimacy: 8.6, P2: Public Services: 8.9, P3: Human Rights: 7.5,
Country: Sudan, Year: 2023, Rank: 7th, Total: 106.19999999999999, S1: Demographic Pressures: 8.8, S2: Refugees and IDPs: 9.6, C3: Group Grievance: 9.3, E3: Human Flight and Brain Drain: 7.5, E2: Economic Inequality: 8.5, E1: Economy: 9.3, P1: State Legitimacy: 9.4, P2: Public Services: 8.6, P3:
Country: Liberia, Year: 2023, Rank: 33rd, Total: 88.89999999999999, S1: Demographic Pressures: 8.4, S2: Refugees and IDPs: 7.4, C3: Group Grievance: 4.6, E3: Human Flight and Brain Drain: 6.7, E2: Economic Inequality: 7.5, E1: Economy: 8.3, P1: State Legitimacy: 6.2, P2: Public Services: 9.1, P3:
Country: Mauritania, Year: 2023, Rank: 37th, Total: 87.0, S1: Demographic Pressures: 8.6, S2: Refugees an

In [27]:
retrieved_text = "\n\n".join(
    [doc["text"] for doc in results]
)

In [28]:
prompt = f"""
You are an Excel analysis assistant.

Answer the question using the context below.

Context:
{retrieved_text}

Question:
{query}

Answer:
"""

In [ ]:
# from transformers import pipeline

# generator = pipeline(
#     "text-generation",
#     model="meta-llama/Llama-3.1-8B-Instruct",
#     device="cuda"
# )

# response = generator(
#     prompt,
#     max_new_tokens=200
# )

# print(response[0]["generated_text"])

Fetching 4 files:   0%|          | 0/4 [08:43<?, ?it/s]


In [3]:
!pip install ollama

In [4]:
!ollama run llama3.1

⠙ ⠹ ⠹ ⠼ ⠴ ⠦ ⠧ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠦ ⠦ ⠇ ⠇ ⠋ ⠙ ⠹ ⠸ ⠸ ⠴ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠹ ⠸ ⠼ ⠼ ⠴ ⠧ ⠧ ⠏ ⠏ ⠋ ⠹ ⠸ ⠼ ⠼ ⠦ ⠧ ⠧ ⠏ ⠏ ⠋ ⠹ ⠸ ⠼ ⠴ ⠴ ⠦ ⠧ ⠏ ⠋ ⠋ ⠙ ⠸ ⠼ ⠼ ⠴ ⠧ ⠧ ⠏ ⠏ ⠋ ⠙ ⠹ ⠸ ⠴ ⠴ ⠧ ⠧ ⠏ ⠋ ⠋ ⠹ ⠸ ⠼ ⠴ ⠴ ⠧ ⠇ ⠇ ⠋ ⠙ ⠹ ⠹ ⠸ >>> Send a message (/? for help)

In [29]:
import ollama

response = ollama.chat(
    model="llama3.1",
    messages=[
        {
            "role": "user",
            
            "content": prompt
        }
    ]
)

final_answer = response["message"]["content"]

print(final_answer)

The rank of Mali is 13th.
